In [ ]:
import pandas as pd

# 2) CSV dosyasını oku ve "df" adlı değişkene koy.
# df = DataFrame, yani Python'daki tablo.
# sep=";"  -> dosyada sütunlar virgül yerine noktalı virgülle ayrılmış
# encoding="utf-8-sig" -> dosyanın başındaki gizli işareti (BOM) temizler,
#                         yoksa ilk sütun adı bozuk gelir
df = pd.read_csv("data_3.csv", sep=";", encoding="utf-8-sig")

# 3) Sütun adlarındaki gizli boşluk ve sekme karakterlerini temizle.
# Bir sütun adında kalan "\t" ileride hata çıkarır (bu dosyada var).
# strip() bir metnin başındaki ve sonundaki boşlukları siler.
df.columns = [c.strip() for c in df.columns]

# 4) Tablonun boyutunu yazdır: (satır sayısı, sütun sayısı)
print(df.shape)

# 5) "Target" sütununda her değerden kaç öğrenci var, say.
print(df["Target"].value_counts())

# 6) Tablonun ilk 5 satırını göster.
# Hücrenin son satırı olduğu için print() yazmadan da ekrana basılır.
df.head()

In [ ]:
# 1) Hiç ders almayan öğrencileri çıkar.
# "Curricular units 1st sem (enrolled)" = 1. dönemde alınan ders sayısı.
# 0 ise "kaldı mı geçti mi" sorusu anlamsız, bu yüzden satırı atıyoruz.
# .copy() -> yeni bir tablo oluşturur, sonraki değişiklikler uyarı vermesin diye
df = df[df["Curricular units 1st sem (enrolled)"] > 0].copy()

# 2) Hedef sütunu oluştur: "basarisiz"
# Alınan ders sayısı > geçilen ders sayısı ise en az 1 dersten kalmış demektir.
# Bu karşılaştırma True/False üretir, .astype(int) bunu 1/0'a çevirir.
# 1 = başarısız, 0 = başarılı
df["basarisiz"] = (
    df["Curricular units 1st sem (enrolled)"]
    > df["Curricular units 1st sem (approved)"]
).astype(int)

# 3) Etiketin dağılımına bak.
# Beklenen: 1 -> 2516, 0 -> 1728 (toplam 4244)
print(df["basarisiz"].value_counts())

# 4) Modele VERMEYECEĞİMİZ sütunları listele.
# - "Target": okulu bırakma bilgisi, dönem sonrası bir sonuç (sızıntı)
# - "sem" geçen her sütun: 1. ve 2. dönem verileri (sızıntı)
# - Nacionality, International, Educational special needs: neredeyse herkes aynı
#   değerde, model bunlardan bir şey öğrenemez
sem_sutunlari = [c for c in df.columns if "sem" in c]
atilacaklar = sem_sutunlari + ["Target", "Nacionality", "International",
                               "Educational special needs"]

# 5) Özellikler (X) ve hedef (y) olarak ayır.
# X = modele göstereceğimiz sorular, y = cevaplar.
# axis=1 -> satır değil sütun silinecek demek
X = df.drop(columns=atilacaklar + ["basarisiz"])
y = df["basarisiz"]

# 6) Sonucu kontrol et.
print("X boyutu:", X.shape)
print("Kalan sütunlar:")
print(list(X.columns))

In [ ]:
# 1) Gerekli araçları içeri aktar.
from sklearn.model_selection import train_test_split          # veriyi bölmek için
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # dönüştürücüler
from sklearn.compose import ColumnTransformer                  # sütun gruplarına farklı dönüşüm uygular

# 2) Veriyi %80 eğitim, %20 test olarak ayır.
# test_size=0.2     -> verinin %20'si teste gider
# stratify=y        -> hem eğitimde hem testte başarısız oranı aynı (~%59) kalır
# random_state=42   -> bölme her çalıştırmada aynı olsun, sonuçlar tekrarlanabilsin
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3) Boyutları ve oranları kontrol et.
print("Eğitim:", X_train.shape, " Test:", X_test.shape)
print("Eğitimde başarısız oranı:", round(y_train.mean(), 3))
print("Testte başarısız oranı  :", round(y_test.mean(), 3))

# 4) Sütunları ikiye ayır.
# Kategorik: sayılar sadece bir KOD (ör. Course=9254 bir bölüm numarası).
# 9254, 171'den "büyük" değildir, bu yüzden bunları sayı gibi ele alamayız.
kategorik = ["Marital status", "Application mode", "Course",
             "Previous qualification", "Mother's qualification",
             "Father's qualification", "Mother's occupation",
             "Father's occupation"]

# Sayısal: geri kalan her şey (yaş, not, işsizlik oranı gibi gerçek sayılar)
sayisal = [c for c in X.columns if c not in kategorik]

# 5) Ön işleme kurallarını tanımla.
# - Kategorikler -> OneHotEncoder: her kategoriye ayrı 0/1 sütunu açar.
#   handle_unknown="ignore": testte eğitimde görülmemiş kod çıkarsa hata verme
#   min_frequency=20: 20'den az öğrencide görülen nadir kodları "diğer" grubuna topla
# - Sayısallar -> StandardScaler: ortalama 0, yayılım 1 olacak şekilde ölçekler.
#   Yaş (17-60) ile GDP (-4 ile 4) aynı ölçeğe gelsin; KNN, SVM ve lojistik regresyon
#   bunu ister.
onisleme = ColumnTransformer([
    ("kat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), kategorik),
    ("say", StandardScaler(), sayisal),
])

In [ ]:
# 1) Gerekli araçlar
from sklearn.pipeline import Pipeline                 # ön işleme + modeli tek zincirde birleştirir
from sklearn.dummy import DummyClassifier             # "aptal" referans model
from sklearn.linear_model import LogisticRegression   # ilk gerçek modelimiz
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# 2) Bir değerlendirme fonksiyonu yazalım ki her modelde aynı kodu tekrar yazmayalım.
def degerlendir(isim, model):
    # fit: modeli EĞİTİM verisiyle eğit (onisleme de burada eğitimden öğrenir)
    model.fit(X_train, y_train)

    # predict: test öğrencileri için 0/1 tahmini
    tahmin = model.predict(X_test)

    # predict_proba: "başarısız olma olasılığı" (0 ile 1 arası); [:,1] = sınıf 1'in olasılığı
    olasilik = model.predict_proba(X_test)[:, 1]

    print("=====", isim, "=====")
    print("Doğruluk (accuracy):", round(accuracy_score(y_test, tahmin), 3))
    print("ROC-AUC            :", round(roc_auc_score(y_test, olasilik), 3))
    print("Karışıklık matrisi (satır=gerçek, sütun=tahmin):")
    print(confusion_matrix(y_test, tahmin))
    print(classification_report(y_test, tahmin, digits=3))

# 3) Referans model: herkese "başarısız" der (çoğunluk sınıfı)
referans = Pipeline([("on", onisleme),
                     ("model", DummyClassifier(strategy="most_frequent"))])
degerlendir("Referans (hep başarısız de)", referans)

# 4) Lojistik regresyon: max_iter=1000 -> öğrenme için en fazla 1000 deneme hakkı
lojistik = Pipeline([("on", onisleme),
                     ("model", LogisticRegression(max_iter=1000))])
degerlendir("Lojistik Regresyon", lojistik)

In [ ]:
# 1) Yeni araçlar
from sklearn.tree import DecisionTreeClassifier                       # Karar Ağacı
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier  # Topluluk modelleri
from sklearn.neighbors import KNeighborsClassifier                    # En yakın komşular
from sklearn.svm import SVC                                           # Destek vektör makinesi
from sklearn.model_selection import cross_val_score, StratifiedKFold  # Çapraz doğrulama
from sklearn.metrics import f1_score, recall_score

# 2) Denenecek modeller (isim: model sözlüğü)
# max_depth=5          -> ağaç en fazla 5 soru derinliğinde, ezberlemesin
# n_estimators=300     -> Random Forest 300 ağaçtan oluşsun
# min_samples_leaf=3   -> her yaprakta en az 3 öğrenci olsun (ezberi azaltır)
# n_neighbors=25       -> KNN 25 en yakın komşuya baksın
# probability=True     -> SVM'den olasılık da alabilelim
# random_state=42      -> sonuçlar her çalıştırmada aynı çıksın
modeller = {
    "Lojistik Regresyon": LogisticRegression(max_iter=1000),
    "Karar Ağacı": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                            random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=25),
    "SVM": SVC(probability=True, random_state=42),
}

# 3) 5 parçalı çapraz doğrulama ayarı (shuffle=True: önce karıştır)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4) Her modeli sırayla eğit ve sonuçları listeye yaz
sonuclar = []
for isim, model in modeller.items():
    # Her modeli aynı ön işlemeyle zincirle
    boru = Pipeline([("on", onisleme), ("model", model)])

    # CV: sadece EĞİTİM verisi üstünde, 5 sınavın ortalama AUC'si
    cv_auc = cross_val_score(boru, X_train, y_train, cv=cv, scoring="roc_auc").mean()

    # Tüm eğitim verisiyle eğit, test verisinde dene
    boru.fit(X_train, y_train)
    tahmin = boru.predict(X_test)
    olasilik = boru.predict_proba(X_test)[:, 1]

    sonuclar.append({
        "Model": isim,
        "CV_AUC": round(cv_auc, 3),
        "Test_AUC": round(roc_auc_score(y_test, olasilik), 3),
        "Doğruluk": round(accuracy_score(y_test, tahmin), 3),
        "F1": round(f1_score(y_test, tahmin), 3),
        # Recall: gerçekten başarısız olanların yüzde kaçını yakaladık?
        "Recall": round(recall_score(y_test, tahmin), 3),
    })

# 5) Tablo yap ve en iyiden en kötüye sırala
tablo = pd.DataFrame(sonuclar).sort_values("Test_AUC", ascending=False)
print(tablo.to_string(index=False))

In [ ]:
!pip install shap

In [ ]:
import numpy as np
import shap
import matplotlib.pyplot as plt

# 1) Random Forest'ı eğit (Adım 5'teki ayarlarla aynı)
rf_boru = Pipeline([
    ("on", onisleme),
    ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                     random_state=42, n_jobs=-1)),
])
rf_boru.fit(X_train, y_train)

# 2) Test verisini, modelin gördüğü haline (ön işlenmiş) çevir.
# OneHot sonrası sütun sayısı 21'den ~102'ye çıkar (her bölüm/meslek kodu ayrı sütun).
Xt = rf_boru.named_steps["on"].transform(X_test)
if hasattr(Xt, "toarray"):      # seyrek (sparse) matris olduysa normal tabloya çevir
    Xt = Xt.toarray()

# 3) Sütun adlarını al ve "kat__" / "say__" ön eklerini temizle
isimler = [n.split("__", 1)[1] for n in rf_boru.named_steps["on"].get_feature_names_out()]
Xt_df = pd.DataFrame(Xt, columns=isimler)

# 4) Random Forest'ın kendi önem puanları (toplamları 1)
onem = pd.Series(rf_boru.named_steps["model"].feature_importances_, index=isimler)
print(onem.sort_values(ascending=False).head(12).round(3))

In [ ]:
%pip install shap

In [ ]:
# 1) SHAP açıklayıcısını Random Forest'a göre kur.
# TreeExplainer: ağaç tabanlı modeller için hızlı ve kesin hesap yapar.
aciklayici = shap.TreeExplainer(rf_boru.named_steps["model"])
shap_degerleri = aciklayici.shap_values(Xt_df)

# 2) Sürüme göre çıktı biçimi değişebilir; "başarısız (1)" sınıfının değerlerini seç.
if isinstance(shap_degerleri, list):
    sv = shap_degerleri[1]
elif shap_degerleri.ndim == 3:
    sv = shap_degerleri[:, :, 1]
else:
    sv = shap_degerleri

# 3) Özet grafiği: her nokta bir öğrenci.
# Sağa kayan nokta = o özellik riski ARTIRIYOR
# Kırmızı = özelliğin değeri yüksek, mavi = düşük
shap.summary_plot(sv, Xt_df, max_display=15)

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 1) Çıkaracağımız 3 makro sütun (öğrenciye değil, kayıt dönemine ait)
makro = ["Unemployment rate", "Inflation rate", "GDP"]

# 2) Verilen sütun listesine göre ön işleme kuran küçük fonksiyon.
# Önceki onisleme sabit sütun listesiyle kurulmuştu; sütunlar azalınca yeniden kurmamız gerek.
def onisleme_kur(sutunlar):
    kat = [c for c in kategorik if c in sutunlar]       # listede kalan kategorikler
    say = [c for c in sutunlar if c not in kat]         # geri kalanlar sayısal
    return ColumnTransformer([
        ("kat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), kat),
        ("say", StandardScaler(), say),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3) İki durumu karşılaştır: makro VAR ve makro YOK
for etiket, sutunlar in [("Makro VAR", list(X.columns)),
                         ("Makro YOK", [c for c in X.columns if c not in makro])]:
    for isim, model in [
        ("Random Forest", RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                 random_state=42, n_jobs=-1)),
        ("Gradient Boosting", GradientBoostingClassifier(random_state=42)),
        ("Lojistik Reg.", LogisticRegression(max_iter=1000)),
    ]:
        boru = Pipeline([("on", onisleme_kur(sutunlar)), ("model", model)])
        cv_auc = cross_val_score(boru, X_train[sutunlar], y_train,
                                 cv=cv, scoring="roc_auc").mean()
        boru.fit(X_train[sutunlar], y_train)
        test_auc = roc_auc_score(y_test, boru.predict_proba(X_test[sutunlar])[:, 1])
        print(f"{etiket:10s} | {isim:18s} | CV AUC: {cv_auc:.3f} | Test AUC: {test_auc:.3f}")

In [ ]:
import joblib   # modeli dosyaya kaydedip geri yüklemek için
import os

# 1) Son modelde kullanacağımız sütunlar: makro 3 sütun hariç hepsi (19 sütun)
son_sutunlar = [c for c in X.columns if c not in makro]

# 2) Son model: ön işleme + Gradient Boosting, tek zincirde.
# Zincir sayesinde siteye ham öğrenci verisi verince dönüşümleri model kendisi yapar.
son_model = Pipeline([
    ("on", onisleme_kur(son_sutunlar)),
    ("model", GradientBoostingClassifier(random_state=42)),
])

# 3) Eğitim verisiyle eğit
son_model.fit(X_train[son_sutunlar], y_train)

# 4) Test verisinde hızlı kontrol (0.72 civarı bekliyoruz)
auc = roc_auc_score(y_test, son_model.predict_proba(X_test[son_sutunlar])[:, 1])
print("Test AUC:", round(auc, 3))

# 5) Modeli VE sütun listesini birlikte dosyaya kaydet.
# Sütun listesini de kaydediyoruz ki site hangi sütunları, hangi sırayla vereceğini bilsin.
joblib.dump({"model": son_model, "sutunlar": son_sutunlar}, "model.joblib")

# 6) Sitede deneme için örnek öğrenci dosyası oluştur.
# Test verisinden 849 öğrenci, her birine 1001'den başlayan bir öğrenci no veriyoruz.
# reset_index(drop=True): eski satır numaralarını at, 0'dan yeniden say
ornek = X_test[son_sutunlar].reset_index(drop=True)
ornek.insert(0, "ogrenci_no", range(1001, 1001 + len(ornek)))
ornek.to_csv("ornek_ogrenciler.csv", index=False)

# 7) Dosyaların nereye kaydedildiğini göster
print("Klasör:", os.getcwd())
print(os.listdir())

In [ ]:
%pip install streamlit

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
import shap
import streamlit as st

# ---------- Sayfa ayarı ----------
st.set_page_config(page_title="Erken Uyarı Sistemi", page_icon="🎓", layout="wide")

# ---------- Model ve sabitler ----------
# cache_resource: model bir kez yüklenir, her tıklamada tekrar yüklenmez
@st.cache_resource
def modeli_yukle():
    paket = joblib.load("model.joblib")
    return paket["model"], paket["sutunlar"]

model, sutunlar = modeli_yukle()

KATEGORIK = ["Marital status", "Application mode", "Course", "Previous qualification",
             "Mother's qualification", "Father's qualification",
             "Mother's occupation", "Father's occupation"]

# Bölüm kodları -> adlar (veri sözlüğüne göre)
BOLUMLER = {171: "Animasyon ve Multimedya Tasarımı", 8014: "Sosyal Hizmet (akşam)",
            9003: "Ziraat Mühendisliği", 9070: "İletişim Tasarımı",
            9085: "Veteriner Hemşireliği", 9119: "Bilişim Mühendisliği",
            9130: "At Yetiştiriciliği", 9147: "İşletme", 9238: "Sosyal Hizmet",
            9254: "Turizm", 9500: "Hemşirelik", 9556: "Ağız ve Diş Sağlığı",
            9670: "Reklamcılık ve Pazarlama", 9773: "Gazetecilik ve İletişim",
            9853: "Temel Eğitim", 9991: "İşletme (akşam)"}

# Sütunların Türkçe adları (grafikte ve nedenlerde görünür)
SUTUN_TR = {"Marital status": "Medeni durum (kod)", "Application mode": "Başvuru türü (kod)",
            "Application order": "Tercih sırası", "Course": "Bölüm",
            "Daytime/evening attendance": "Gündüz/akşam eğitimi",
            "Previous qualification": "Önceki eğitim türü (kod)",
            "Previous qualification (grade)": "Önceki eğitim notu",
            "Mother's qualification": "Anne eğitimi (kod)", "Father's qualification": "Baba eğitimi (kod)",
            "Mother's occupation": "Anne mesleği (kod)", "Father's occupation": "Baba mesleği (kod)",
            "Admission grade": "Giriş notu", "Displaced": "Yerinden edilmiş",
            "Debtor": "Borç durumu", "Tuition fees up to date": "Harç durumu",
            "Gender": "Cinsiyet", "Scholarship holder": "Burs durumu",
            "Age at enrollment": "Kayıt yaşı"}

# Bir sütunun değerini okunabilir Türkçe cümleye çevirir
def neden_metni(sutun, deger):
    if float(deger).is_integer():
        deger = int(deger)
    if sutun == "Scholarship holder":         return "Burslu değil" if deger == 0 else "Burslu"
    if sutun == "Tuition fees up to date":    return "Harcı ödenmemiş" if deger == 0 else "Harcı ödenmiş"
    if sutun == "Debtor":                     return "Borçlu" if deger == 1 else "Borcu yok"
    if sutun == "Gender":                     return "Erkek" if deger == 1 else "Kadın"
    if sutun == "Daytime/evening attendance": return "Gündüz eğitimi" if deger == 1 else "Akşam eğitimi"
    if sutun == "Displaced":                  return "Yerinden edilmiş" if deger == 1 else "Yerinden edilmemiş"
    if sutun == "Course":                     return "Bölüm: " + BOLUMLER.get(deger, str(deger))
    return f"{SUTUN_TR.get(sutun, sutun)}: {deger}"

# OneHot sütun adını ("kat__Course_9500") asıl sütuna ("Course") çevirir
def asil_sutun(ad):
    ad = ad.split("__", 1)[1]
    for k in sorted(KATEGORIK, key=len, reverse=True):
        if ad.startswith(k + "_"):
            return k
    return ad

# cache_data: aynı veri için hesap tekrarlanmaz (eşik kaydırıcısı hızlı çalışır)
@st.cache_data
def risk_hesapla(df, top=3):
    X_ogr = df[sutunlar]
    on = model.named_steps["on"]
    gb = model.named_steps["model"]
    Xt = on.transform(X_ogr)
    Xt = Xt.toarray() if hasattr(Xt, "toarray") else Xt
    shap_vals = shap.TreeExplainer(gb).shap_values(Xt)
    eslesme = [asil_sutun(n) for n in on.get_feature_names_out()]
    shap_df = pd.DataFrame(shap_vals).T.groupby(eslesme).sum().T
    risk = model.predict_proba(X_ogr)[:, 1]
    nedenler = []
    for i in range(len(X_ogr)):
        en_buyuk = shap_df.iloc[i].sort_values(ascending=False).head(top)
        en_buyuk = en_buyuk[en_buyuk > 0]
        nedenler.append([neden_metni(s, X_ogr.iloc[i][s]) for s in en_buyuk.index])
    sonuc = pd.DataFrame({"Öğrenci No": df["ogrenci_no"].values,
                          "Risk (%)": (risk * 100).round(1)})
    for k in range(top):
        sonuc[f"Neden {k+1}"] = [n[k] if len(n) > k else "" for n in nedenler]
    shap_df.index = sonuc["Öğrenci No"]
    return sonuc, shap_df

# ---------- Başlık ----------
st.title("🎓 Akademik Başarısızlık Erken Uyarı Sistemi")
st.caption("Dönem başındaki kayıt bilgilerinden, 1. dönemde en az bir dersten kalma riskini tahmin eder.")

# ---------- Yan panel ----------
with st.sidebar:
    st.header("Ayarlar")
    yuklenen = st.file_uploader("Öğrenci CSV dosyası yükle", type="csv")
    ornek_kullan = st.checkbox("Örnek veriyi kullan", value=yuklenen is None)
    esik = st.slider("Risk eşiği (%)", 50, 95, 70, 5,
                     help="Bu değerin üstündeki öğrenciler 'riskli' sayılır.")

if yuklenen is not None:
    ogrenciler = pd.read_csv(yuklenen)
elif ornek_kullan:
    ogrenciler = pd.read_csv("ornek_ogrenciler.csv")
else:
    st.info("Soldan bir CSV yükleyin veya örnek veriyi seçin.")
    st.stop()

# CSV'de gerekli sütunlar var mı kontrol et
eksik = [c for c in ["ogrenci_no"] + sutunlar if c not in ogrenciler.columns]
if eksik:
    st.error("CSV dosyasında şu sütunlar eksik: " + ", ".join(eksik))
    st.stop()

sonuc, shap_df = risk_hesapla(ogrenciler)
riskli = sonuc[sonuc["Risk (%)"] >= esik].sort_values("Risk (%)", ascending=False)

# ---------- Özet kutuları ----------
s1, s2, s3 = st.columns(3)
s1.metric("Toplam öğrenci", len(sonuc))
s2.metric(f"Riskli öğrenci (≥ %{esik})", len(riskli))
s3.metric("Riskli oranı", f"%{100 * len(riskli) / len(sonuc):.1f}")

# ---------- Riskli öğrenci tablosu ----------
st.subheader("Riskli öğrenciler")
st.dataframe(riskli, width="stretch", hide_index=True,
             column_config={"Risk (%)": st.column_config.ProgressColumn(
                 "Risk (%)", min_value=0, max_value=100, format="%.1f")})
st.download_button("Listeyi CSV olarak indir",
                   riskli.to_csv(index=False).encode("utf-8-sig"),
                   file_name="riskli_ogrenciler.csv", mime="text/csv")

# ---------- Öğrenci detayı ----------
st.subheader("Öğrenci detayı")
if len(riskli) == 0:
    st.success("Bu eşikte riskli öğrenci yok.")
else:
    secilen = st.selectbox("Öğrenci seç", riskli["Öğrenci No"])
    satir = sonuc[sonuc["Öğrenci No"] == secilen].iloc[0]
    st.metric("Risk", f"%{satir['Risk (%)']:.1f}")
    st.write("**Başlıca risk faktörleri:**",
             " • ".join(v for v in [satir["Neden 1"], satir["Neden 2"], satir["Neden 3"]] if v))
    etki = shap_df.loc[secilen].rename(index=SUTUN_TR).sort_values(ascending=False).head(8)
    st.caption("Faktör etkileri (pozitif = riski artırır, negatif = azaltır)")
    st.bar_chart(etki)

# ---------- Uyarı ----------
st.warning("Bu sistem karar verici değil, destek amaçlıdır. Listelenen faktörler modelin tahminde "
           "dayandığı örüntülerdir; riskin kanıtlanmış nedeni değildir. Öğrencilere destek "
           "sunulacak kişileri önceliklendirmek için kullanılmalıdır.")

In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)   # gizli ayar klasörü
with open(".streamlit/config.toml", "w") as f:
    f.write('''[theme]
base = "light"
primaryColor = "#1F4E79"
backgroundColor = "#F5F7FA"
secondaryBackgroundColor = "#FFFFFF"
textColor = "#1B2A3A"
''')
print("Tema dosyası yazıldı")

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
import shap
import streamlit as st

st.set_page_config(page_title="Öğrenci Başarı Erken Uyarı Sistemi", page_icon="🎓", layout="wide")

# ====================== GÖRÜNÜM (CSS) ======================
st.markdown("""
<style>
#MainMenu, footer {visibility: hidden;}
.block-container {padding-top: 1.5rem; max-width: 1250px;}
.banner {background: linear-gradient(120deg, #12395F 0%, #1F6FB2 100%); color: #fff;
         padding: 26px 32px; border-radius: 14px; margin-bottom: 18px;}
.banner .b1 {font-size: 13px; letter-spacing: 2px; text-transform: uppercase; opacity: .85;}
.banner .b2 {font-size: 30px; font-weight: 700; margin: 4px 0;}
.banner .b3 {font-size: 15px; opacity: .92;}
.kpi {background: #fff; padding: 14px 18px; border-radius: 10px; box-shadow: 0 1px 5px rgba(20,40,70,.12);}
.kpi .kb {font-size: 13px; color: #5B6B7B; font-weight: 600;}
.kpi .kd {font-size: 30px; font-weight: 700; color: #12395F; line-height: 1.2;}
.kpi .ka {font-size: 12px; color: #7B8A99;}
.kutu {background: #fff; padding: 16px 20px; border-radius: 10px; box-shadow: 0 1px 5px rgba(20,40,70,.10);}
.etiket {display: inline-block; padding: 3px 10px; border-radius: 12px; background: #E8F0F8;
         color: #12395F; font-size: 13px; margin: 2px 4px 2px 0;}
</style>
""", unsafe_allow_html=True)

# Özet kartı çizen küçük yardımcı fonksiyon
def kpi(baslik, deger, alt="", renk="#1F4E79"):
    st.markdown(f'<div class="kpi" style="border-left:6px solid {renk}"><div class="kb">{baslik}</div>'
                f'<div class="kd">{deger}</div><div class="ka">{alt}</div></div>', unsafe_allow_html=True)

# ====================== MODEL VE SABİTLER ======================
@st.cache_resource
def modeli_yukle():
    paket = joblib.load("model.joblib")
    return paket["model"], paket["sutunlar"]

model, sutunlar = modeli_yukle()

KATEGORIK = ["Marital status", "Application mode", "Course", "Previous qualification",
             "Mother's qualification", "Father's qualification",
             "Mother's occupation", "Father's occupation"]

BOLUMLER = {171: "Animasyon ve Multimedya Tasarımı", 8014: "Sosyal Hizmet (akşam)",
            9003: "Ziraat Mühendisliği", 9070: "İletişim Tasarımı",
            9085: "Veteriner Hemşireliği", 9119: "Bilişim Mühendisliği",
            9130: "At Yetiştiriciliği", 9147: "İşletme", 9238: "Sosyal Hizmet",
            9254: "Turizm", 9500: "Hemşirelik", 9556: "Ağız ve Diş Sağlığı",
            9670: "Reklamcılık ve Pazarlama", 9773: "Gazetecilik ve İletişim",
            9853: "Temel Eğitim", 9991: "İşletme (akşam)"}

SUTUN_TR = {"Marital status": "Medeni durum (kod)", "Application mode": "Başvuru türü (kod)",
            "Application order": "Tercih sırası", "Course": "Bölüm",
            "Daytime/evening attendance": "Gündüz/akşam eğitimi",
            "Previous qualification": "Önceki eğitim türü (kod)",
            "Previous qualification (grade)": "Önceki eğitim notu",
            "Mother's qualification": "Anne eğitimi (kod)", "Father's qualification": "Baba eğitimi (kod)",
            "Mother's occupation": "Anne mesleği (kod)", "Father's occupation": "Baba mesleği (kod)",
            "Admission grade": "Giriş notu", "Displaced": "Yerinden edilmiş",
            "Debtor": "Borç durumu", "Tuition fees up to date": "Harç durumu",
            "Gender": "Cinsiyet", "Scholarship holder": "Burs durumu",
            "Age at enrollment": "Kayıt yaşı"}

# Risk faktörüne göre önerilen destek adımı (kural tabanlı, karar değil)
ONERI = {"Tuition fees up to date": "Mali işlerle harç yapılandırma / mali destek görüşmesi",
         "Debtor": "Mali işlerle borç yapılandırma görüşmesi",
         "Scholarship holder": "Burs ve destek programları hakkında bilgilendirme",
         "Age at enrollment": "Çalışan/yetişkin öğrenciler için esnek program ve danışmanlık",
         "Admission grade": "Akademik hazırlık programı, mentörlük ve ders desteği",
         "Previous qualification (grade)": "Akademik hazırlık programı ve ders desteği",
         "Course": "Bölüm danışmanıyla erken dönem görüşmesi",
         "Application mode": "Oryantasyon ve uyum programına yönlendirme",
         "Application order": "Bölüm tercihi/uyum görüşmesi (kariyer danışmanlığı)",
         "Displaced": "Barınma ve uyum destek birimine yönlendirme",
         "Daytime/evening attendance": "Akşam programı öğrencileri için esnek ders ve danışmanlık"}

def neden_metni(sutun, deger):
    if float(deger).is_integer():
        deger = int(deger)
    if sutun == "Scholarship holder":         return "Burslu değil" if deger == 0 else "Burslu"
    if sutun == "Tuition fees up to date":    return "Harcı ödenmemiş" if deger == 0 else "Harcı ödenmiş"
    if sutun == "Debtor":                     return "Borçlu" if deger == 1 else "Borcu yok"
    if sutun == "Gender":                     return "Erkek" if deger == 1 else "Kadın"
    if sutun == "Daytime/evening attendance": return "Gündüz eğitimi" if deger == 1 else "Akşam eğitimi"
    if sutun == "Displaced":                  return "Yerinden edilmiş" if deger == 1 else "Yerinden edilmemiş"
    if sutun == "Course":                     return "Bölüm: " + BOLUMLER.get(deger, str(deger))
    return f"{SUTUN_TR.get(sutun, sutun)}: {deger}"

def asil_sutun(ad):
    ad = ad.split("__", 1)[1]
    for k in sorted(KATEGORIK, key=len, reverse=True):
        if ad.startswith(k + "_"):
            return k
    return ad

@st.cache_data
def risk_hesapla(df, top=3):
    X_ogr = df[sutunlar]
    on = model.named_steps["on"]
    gb = model.named_steps["model"]
    Xt = on.transform(X_ogr)
    Xt = Xt.toarray() if hasattr(Xt, "toarray") else Xt
    shap_vals = shap.TreeExplainer(gb).shap_values(Xt)
    eslesme = [asil_sutun(n) for n in on.get_feature_names_out()]
    shap_df = pd.DataFrame(shap_vals).T.groupby(eslesme).sum().T
    risk = model.predict_proba(X_ogr)[:, 1]
    nedenler, anahtarlar = [], []
    for i in range(len(X_ogr)):
        en_buyuk = shap_df.iloc[i].sort_values(ascending=False).head(top)
        en_buyuk = en_buyuk[en_buyuk > 0]
        nedenler.append([neden_metni(s, X_ogr.iloc[i][s]) for s in en_buyuk.index])
        anahtarlar.append(list(en_buyuk.index))
    sonuc = pd.DataFrame({
        "Öğrenci No": df["ogrenci_no"].values,
        "Bölüm": df["Course"].map(lambda k: BOLUMLER.get(int(k), str(int(k)))).values,
        "Risk (%)": (risk * 100).round(1)})
    for k in range(top):
        sonuc[f"Neden {k+1}"] = [n[k] if len(n) > k else "" for n in nedenler]
    shap_df.index = sonuc["Öğrenci No"]
    return sonuc, shap_df, dict(zip(sonuc["Öğrenci No"], anahtarlar))

# ====================== YAN PANEL ======================
with st.sidebar:
    st.markdown("### ⚙️ Ayarlar")
    kurum = st.text_input("Kurum adı", "Örnek Üniversitesi")
    st.markdown("**Veri kaynağı**")
    yuklenen = st.file_uploader("Öğrenci CSV dosyası", type="csv", label_visibility="collapsed")
    ornek_kullan = st.checkbox("Örnek veriyi kullan", value=yuklenen is None)
    st.markdown("**Risk seviyeleri**")
    orta_esik, yuksek_esik = st.slider("Orta ve yüksek risk eşikleri (%)", 30, 95, (50, 75), 5,
        help="Altı: Düşük  |  Aradaki: Orta  |  Üstü: Yüksek")
    st.caption("Şablon CSV'yi 'Model ve Yöntem' sekmesinden indirebilirsiniz.")

st.markdown(f'<div class="banner"><div class="b1">{kurum}</div>'
            '<div class="b2">🎓 Öğrenci Başarı Erken Uyarı Sistemi</div>'
            '<div class="b3">Dönem başı kayıt bilgilerinden, 1. dönemde en az bir dersten kalma riskini '
            'önceden tahmin eder ve destek verilecek öğrencileri önceliklendirir.</div></div>',
            unsafe_allow_html=True)

if yuklenen is not None:
    ogrenciler = pd.read_csv(yuklenen)
elif ornek_kullan:
    ogrenciler = pd.read_csv("ornek_ogrenciler.csv")
else:
    st.info("Soldaki panelden bir CSV dosyası yükleyin veya örnek veriyi seçin.")
    st.stop()

eksik = [c for c in ["ogrenci_no"] + sutunlar if c not in ogrenciler.columns]
if eksik:
    st.error("CSV dosyasında şu sütunlar eksik: " + ", ".join(eksik))
    st.stop()
if ogrenciler["ogrenci_no"].duplicated().any():
    st.error("CSV dosyasında tekrar eden öğrenci numaraları var. Her öğrenci bir kez yer almalı.")
    st.stop()

sonuc, shap_df, anahtarlar = risk_hesapla(ogrenciler)
sonuc["Seviye"] = pd.cut(sonuc["Risk (%)"], [-1, orta_esik - 1e-9, yuksek_esik - 1e-9, 101],
                         labels=["🟢 Düşük", "🟠 Orta", "🔴 Yüksek"])

# Bölüm filtresi (veri yüklendikten sonra kurulur)
with st.sidebar:
    tum_bolumler = sorted(sonuc["Bölüm"].unique())
    secili = st.multiselect("Bölüm filtresi", tum_bolumler, default=tum_bolumler)
gorunen = sonuc[sonuc["Bölüm"].isin(secili)]
if gorunen.empty:
    st.warning("Seçili bölümlerde öğrenci yok.")
    st.stop()

sekme1, sekme2, sekme3, sekme4 = st.tabs(["📊 Genel Bakış", "🚨 Riskli Öğrenciler",
                                           "👤 Öğrenci Detayı", "ℹ️ Model ve Yöntem"])

# ====================== SEKME 1: GENEL BAKIŞ ======================
with sekme1:
    n = len(gorunen)
    yuksek_n = (gorunen["Seviye"] == "🔴 Yüksek").sum()
    orta_n = (gorunen["Seviye"] == "🟠 Orta").sum()
    dusuk_n = (gorunen["Seviye"] == "🟢 Düşük").sum()
    c1, c2, c3, c4 = st.columns(4)
    with c1: kpi("Toplam öğrenci", f"{n:,}".replace(",", "."), "Analiz edilen kayıt")
    with c2: kpi("Yüksek risk", yuksek_n, f"%{100*yuksek_n/n:.1f} · öncelikli destek", "#C0392B")
    with c3: kpi("Orta risk", orta_n, f"%{100*orta_n/n:.1f} · izlenmeli", "#E67E22")
    with c4: kpi("Ortalama risk", f"%{gorunen['Risk (%)'].mean():.1f}", f"Düşük risk: {dusuk_n} öğrenci", "#27AE60")
    st.write("")
    g1, g2 = st.columns(2)
    with g1:
        st.markdown("**Risk seviyesi dağılımı**")
        dagilim = gorunen["Seviye"].value_counts().reindex(["🟢 Düşük", "🟠 Orta", "🔴 Yüksek"]).fillna(0)
        st.bar_chart(dagilim)
    with g2:
        st.markdown("**Bölüme göre ortalama risk (%)**")
        bolum_risk = gorunen.groupby("Bölüm")["Risk (%)"].mean().sort_values().round(1)
        st.bar_chart(bolum_risk, horizontal=True)
    st.markdown("**Yüksek riskli öğrencilerde en sık görülen risk faktörleri**")
    sayac = {}
    for no in gorunen.loc[gorunen["Seviye"] == "🔴 Yüksek", "Öğrenci No"]:
        for s in anahtarlar[no]:
            sayac[SUTUN_TR.get(s, s)] = sayac.get(SUTUN_TR.get(s, s), 0) + 1
    if sayac:
        st.bar_chart(pd.Series(sayac).sort_values(), horizontal=True)
    else:
        st.caption("Bu eşikte yüksek riskli öğrenci yok.")

# ====================== SEKME 2: RİSKLİ ÖĞRENCİLER ======================
with sekme2:
    f1, f2 = st.columns([2, 1])
    seviyeler = f1.multiselect("Risk seviyesi", ["🔴 Yüksek", "🟠 Orta", "🟢 Düşük"],
                               default=["🔴 Yüksek"])
    aranan = f2.text_input("Öğrenci no ara")
    liste = gorunen[gorunen["Seviye"].isin(seviyeler)]
    if aranan.strip():
        liste = liste[liste["Öğrenci No"].astype(str).str.contains(aranan.strip())]
    liste = liste.sort_values("Risk (%)", ascending=False)[
        ["Öğrenci No", "Bölüm", "Risk (%)", "Seviye", "Neden 1", "Neden 2", "Neden 3"]]
    st.caption(f"{len(liste)} öğrenci listeleniyor")
    st.dataframe(liste, width="stretch", hide_index=True, height=520,
                 column_config={"Risk (%)": st.column_config.ProgressColumn(
                     "Risk (%)", min_value=0, max_value=100, format="%.1f")})
    st.download_button("⬇️ Listeyi CSV olarak indir", liste.to_csv(index=False).encode("utf-8-sig"),
                       file_name="riskli_ogrenciler.csv", mime="text/csv")

# ====================== SEKME 3: ÖĞRENCİ DETAYI ======================
with sekme3:
    sirali = gorunen.sort_values("Risk (%)", ascending=False)
    risk_sozluk = dict(zip(sirali["Öğrenci No"], sirali["Risk (%)"]))
    secilen = st.selectbox("Öğrenci seçin (riskten düşüğe sıralı)", list(risk_sozluk),
                           format_func=lambda no: f"{no}  —  %{risk_sozluk[no]:.1f}")
    satir = sonuc[sonuc["Öğrenci No"] == secilen].iloc[0]
    d1, d2, d3 = st.columns(3)
    with d1: kpi("Öğrenci no", satir["Öğrenci No"])
    with d2: kpi("Risk", f"%{satir['Risk (%)']:.1f}", str(satir["Seviye"]))
    with d3: kpi("Bölüm", "", satir["Bölüm"])
    st.write("")
    sol, sag = st.columns(2)
    with sol:
        st.markdown("**Başlıca risk faktörleri**")
        nedenler = [v for v in [satir["Neden 1"], satir["Neden 2"], satir["Neden 3"]] if v]
        if nedenler:
            st.markdown("".join(f'<span class="etiket">{v}</span>' for v in nedenler), unsafe_allow_html=True)
        else:
            st.caption("Riski belirgin şekilde artıran bir faktör bulunmuyor.")
        st.markdown("**Önerilen destek adımları**")
        oneriler = [ONERI.get(s, "Akademik danışman görüşmesi") for s in anahtarlar[secilen]]
        for o in dict.fromkeys(oneriler):
            st.markdown(f"- {o}")
        if not oneriler:
            st.markdown("- Rutin izleme yeterli")
        st.caption("Öneriler kural tabanlıdır; nihai karar ilgili birimlere aittir.")
    with sag:
        st.markdown("**Faktör etkileri** (pozitif: riski artırır, negatif: azaltır)")
        etki = shap_df.loc[secilen].rename(index=SUTUN_TR).sort_values().tail(8)
        st.bar_chart(etki, horizontal=True)

# ====================== SEKME 4: MODEL VE YÖNTEM ======================
with sekme4:
    m1, m2, m3 = st.columns(3)
    with m1: kpi("Algoritma", "", "Gradient Boosting + SHAP açıklamaları")
    with m2: kpi("Test başarımı (ROC-AUC)", "0,72", "Çapraz doğrulama: 0,73")
    with m3: kpi("Kullanılan özellik", len(sutunlar), "Yalnızca dönem başı kayıt bilgileri")
    st.write("")
    st.markdown("""
**Amaç:** Öğrencinin 1. dönemde aldığı derslerden en az birinden kalma olasılığını, dönem başında
bilinen bilgilerle tahmin etmek.

**Hedef tanımı:** 1. dönemde alınan ders sayısı geçilen ders sayısından fazlaysa öğrenci "başarısız" sayılır.

**Modele giren bilgiler:** Bölüm, giriş ve önceki eğitim notu, kayıt yaşı, burs, harç ve borç durumu, başvuru türü,
aile bilgileri vb. Dönem içi notlar ve makroekonomik göstergeler **kullanılmaz**.

**Açıklanabilirlik:** Her öğrenci için SHAP yöntemiyle riski artıran başlıca faktörler gösterilir.
""")
    k1, k2 = st.columns(2)
    with k1:
        st.markdown('<div class="kutu"><b>⚠️ Sınırlılıklar</b><ul>'
                    '<li>Model orta düzeyde ayırt edicidir (AUC ≈ 0,72); kesin hüküm değil, öncelik sıralaması üretir.</li>'
                    '<li>Eğitim verisi tek bir ülkedeki açık bir araştırma veri setidir; kendi kurumunuzun verisiyle yeniden eğitilmeli ve doğrulanmalıdır.</li>'
                    '<li>Faktörler ilişkiyi gösterir, nedensellik kanıtı değildir.</li>'
                    '<li>"Başarısız" tanımı sıkıdır (1 dersten kalmak); öğrencilerin yarıdan fazlası bu gruba girebilir.</li>'
                    '</ul></div>', unsafe_allow_html=True)
    with k2:
        st.markdown('<div class="kutu"><b>🔒 Veri gizliliği ve etik</b><ul>'
                    '<li>Bu uygulama yüklenen veriyi diske kaydetmez; yalnızca oturum süresince bellekte işler.</li>'
                    '<li>Gerçek öğrenci verisi için kurum içi sunucu ve KVKK uyumu gerekir; herkese açık bir bağlantıya yüklenmemelidir.</li>'
                    '<li>Cinsiyet ve yaş gibi kişisel özelliklerin kullanımı adalet açısından ayrıca değerlendirilmelidir.</li>'
                    '<li>Sonuçlar cezalandırma için değil, destek sunmak için kullanılmalıdır.</li>'
                    '</ul></div>', unsafe_allow_html=True)
    st.write("")
    sablon = pd.DataFrame(columns=["ogrenci_no"] + sutunlar)
    st.download_button("⬇️ Şablon CSV'yi indir (gerekli sütunlar)",
                       sablon.to_csv(index=False).encode("utf-8-sig"),
                       file_name="sablon_ogrenci_verisi.csv", mime="text/csv")

In [ ]:
import joblib
import os

# 1) Çıkarılacak sütunlar: 3 makro sütun + harç durumu + borç durumu
cikar = makro + ["Tuition fees up to date", "Debtor"]

# 2) Kalan sütunlar (16 tane olmalı)
son_sutunlar = [c for c in X.columns if c not in cikar]
print("Sütun sayısı:", len(son_sutunlar))

# 3) Aynı model, yeni sütun listesiyle
son_model = Pipeline([
    ("on", onisleme_kur(son_sutunlar)),
    ("model", GradientBoostingClassifier(random_state=42)),
])
son_model.fit(X_train[son_sutunlar], y_train)

# 4) Test kontrolü (0.70-0.71 arası bekliyoruz)
auc = roc_auc_score(y_test, son_model.predict_proba(X_test[son_sutunlar])[:, 1])
print("Test AUC:", round(auc, 3))

# 5) Model ve örnek öğrenci dosyasının üzerine yaz
joblib.dump({"model": son_model, "sutunlar": son_sutunlar}, "model.joblib")
ornek = X_test[son_sutunlar].reset_index(drop=True)
ornek.insert(0, "ogrenci_no", range(1001, 1001 + len(ornek)))
ornek.to_csv("ornek_ogrenciler.csv", index=False)
print("Kaydedildi:", os.listdir())

In [ ]:
metin = open("app.py", encoding="utf-8").read()

# "Modele giren bilgiler" cümlesi: harç ve borç ifadesini kaldır
metin = metin.replace("burs, harç ve borç durumu", "burs durumu")
metin = metin.replace("burs ve borç durumu", "burs durumu")

# Başarım kartı
metin = metin.replace('"0,72", "Çapraz doğrulama: 0,73"', '"0,70", "Çapraz doğrulama: 0,71"')
metin = metin.replace('"0,71", "Çapraz doğrulama: 0,72"', '"0,70", "Çapraz doğrulama: 0,71"')

# Sınırlılıklar maddesi
metin = metin.replace("AUC ≈ 0,72", "AUC ≈ 0,70")
metin = metin.replace("AUC ≈ 0,71", "AUC ≈ 0,70")

open("app.py", "w", encoding="utf-8").write(metin)
print("app.py güncellendi")

In [ ]:
# app.py'deki bölüm sözlüğüne eksik 33 kodunu ekle
metin = open("app.py", encoding="utf-8").read()
metin = metin.replace('BOLUMLER = {171:',
                      'BOLUMLER = {33: "Biyoyakıt Üretim Teknolojileri", 171:')
open("app.py", "w", encoding="utf-8").write(metin)
print("Güncellendi:", '33: "Biyoyakıt' in metin)

In [ ]:
import sklearn, shap, streamlit, pandas, numpy, joblib

# Senin bilgisayarında kurulu olan SÜRÜMLERİ otomatik yazar
satirlar = [
    f"scikit-learn=={sklearn.__version__}",
    f"shap=={shap.__version__}",
    f"streamlit=={streamlit.__version__}",
    f"pandas=={pandas.__version__}",
    f"numpy=={numpy.__version__}",
    f"joblib=={joblib.__version__}",
]
open("requirements.txt", "w").write("\n".join(satirlar))
print("\n".join(satirlar))